# Boltz2-Notebook: Diffusion-Based Protein–Ligand Structure Prediction & Affinity Analysis

**AI-powered structure prediction and binding affinity analysis** — Interactive Jupyter notebook for protein structure prediction, ligand binding, and multi-chain complex modeling using the Boltz2 diffusion model.

![Python](https://img.shields.io/badge/Python-3.12-blue?logo=python)
![CUDA](https://img.shields.io/badge/CUDA-Enabled-green?logo=nvidia)
![Boltz2](https://img.shields.io/badge/Model-Boltz2-purple)
![Platform](https://img.shields.io/badge/Platform-Colab%20|%20Linux-lightgrey?logo=googlecolab)
![License](https://img.shields.io/badge/License-MIT-orange)
![Status](https://img.shields.io/badge/Status-Production-success)

---

## Available Notebooks

**V2.0.0 (Latest)** — Advanced Biomolecular Modeling  
Multi-entity structures, DNA/RNA, advanced constraints, custom MSA, cyclic peptides, template guidance, affinity prediction  
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AtharvaTilewale/boltz2-notebook/blob/main/colab/v2/Boltz2_V2.0.0_beta.ipynb)

**V1.0.0 (Stable) [This Notebook]** — Production Ready  
Multi-chain proteins, protein-ligand binding, affinity analysis, 3D visualization  
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AtharvaTilewale/boltz2-notebook/blob/main/colab/v1/Boltz2_v1.0.0.ipynb)

**Batch v1.0 (Experimental)** — High-Throughput Screening  
CSV/FASTA batch processing, automated ranking, large-scale workflows  
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AtharvaTilewale/boltz2-notebook/blob/main/batch/Boltz2_Batch_v1_beta.ipynb)

---

## Overview

**Boltz2-Notebook** is an interactive Google Colab platform that integrates the Boltz2 deep learning model for diffusion-based structure prediction. It provides a fully guided workflow with graphical input, 3D visualization, and automated analysis — no GPU installation or command-line setup required.

---

## Tutorial / How to Use

For step-by-step protocols and practical examples, see:  
[Use Cases Guide](https://atharvatilewale.github.io/boltz2-notebook/docs/use-cases.html)

---

## Workflow

1. **Input** - Provide protein sequence(s), ligands, and optional advanced entities.
2. **Configure** - System generates YAML configuration.
3. **Predict** - MSA generation and structure prediction via diffusion model.
4. **Analyze** - Extract confidence metrics (pLDDT, PAE) and affinity scores.
5. **Visualize** - Interactive 3D structure viewer and confidence plots.
6. **Export** - Download results or save to Google Drive.

**Typical runtime:** 2-10 minutes on T4 GPU

> ----------------**Important Note**----------------<br>
> <br>**Colab Runtime & Python Compatibility:**  
> Due to recent changes in Colab, Colab is using Python 3.13, which makes dependencies installation much slower and takes a lot of time, some are incompatible as well. We recommend to switch to runtime version 2026.7 before running.  
>  
> **How to switch runtime version & enable GPU:**  
> 1. In Colab top menu, go to **Runtime > Change runtime type**.  
> 2. Under **Hardware accelerator**, select **GPU** (T4 or higher).  
> 3. Under **Runtime version**, select **2026.7** (Python 3.12).  
> 4. Click **Save** and **Connect** (top-right) before running any cells.  
> ----------------------------------------------------


In [ ]:
# @title Install Dependencies and Boltz2 with CUDA support
# @markdown **Setup Settings:**
# @markdown - **FORCE_REINSTALL**: Force full reinstallation even if already installed in this runtime.
FORCE_REINSTALL = False # @param {type:"boolean"}

import sys
import os
import subprocess
import threading
import time
import shutil

os.chdir("/content/")
os.environ["FORCE_REINSTALL"] = "1" if FORCE_REINSTALL else "0"

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

# ---------------- RUNTIME & PYTHON VERSION CHECK ----------------
py_major, py_minor = sys.version_info.major, sys.version_info.minor
colab_release = os.environ.get("COLAB_RELEASE_TAG", "Not detected / Local")

print(f"{Color.CYAN}[i] Python version:{Color.RESET} {sys.version.split()[0]}")
if "COLAB_RELEASE_TAG" in os.environ:
    print(f"{Color.CYAN}[i] Colab runtime tag:{Color.RESET} {colab_release}")

if py_major == 3 and py_minor == 12:
    print(f"{Color.GREEN}[✔] Python 3.12 runtime verified (Optimal compatibility).{Color.RESET}")
else:
    print(f"{Color.RED}[!] WARNING: Python {py_major}.{py_minor} detected (Python 3.12 is recommended)!{Color.RESET}")
    print(f"{Color.YELLOW}Due to recent changes in Colab, Colab is using Python 3.13, which makes dependencies installation much slower and takes a lot of time, some are incompatible as well.{Color.RESET}")
    print(f"{Color.YELLOW}We recommend to switch to runtime version 2026.7 before running.{Color.RESET}")
    print(f"{Color.CYAN}To switch: Runtime > Change runtime type > Runtime version: select 2026.7{Color.RESET}\n")

# ---------------- GPU CHECK ----------------
import torch
print(f"{Color.CYAN}[i] Checking GPU availability...{Color.RESET}")
if not torch.cuda.is_available():
    print(f"{Color.RED}[✘] No GPU detected!{Color.RESET}")
    print(f"{Color.YELLOW}Please change runtime to 'GPU' (T4 or higher).{Color.RESET}")
    print(f"{Color.CYAN}Runtime > Change Runtime Type > Select any available GPU from Hardware Accelerator.{Color.RESET}")
    sys.exit(1)
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{Color.GREEN}[✔] GPU detected:{Color.RESET} {gpu_name}")

repo_dirs = ["boltz2-notebook"]

def loader(msg, stop_event):
    symbols = ["-", "\\", "|", "/"]
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f"\r[{symbols[i % len(symbols)]}] {msg}   ")
        sys.stdout.flush()
        time.sleep(0.1)
        i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 10) + "\r")

if not os.path.exists("/content/boltz2-notebook") or FORCE_REINSTALL:
    if os.path.exists("/content/boltz2-notebook"):
        shutil.rmtree("/content/boltz2-notebook")
    stop_event = threading.Event()
    t = threading.Thread(target=loader, args=(f"{Color.CYAN}Cloning Notebook Modules...{Color.RESET}", stop_event))
    t.start()
    try:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AtharvaTilewale/boltz2-notebook.git"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        stop_event.set()
        t.join()
        print(f"[{Color.GREEN}✔{Color.RESET}] Notebook modules cloned successfully.")
    except Exception as e:
        stop_event.set()
        t.join()
        print(f"[{Color.RED}✘{Color.RESET}] Boltz-Notebook clone failed: {e}")
        raise
else:
    print(f"[{Color.GREEN}✔{Color.RESET}] Using existing notebook modules.")

%run /content/boltz2-notebook/scripts/v1/setup.py

if all_success:
    print(f"{Color.GREEN}All steps completed successfully.{Color.RESET}")


In [ ]:

# @title Download CCD Dataset and Test Boltz2
import sys
import threading
import time
import os

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

def loader(msg, stop_event):
    symbols = ["-", "\\", "|", "/"]
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f"\r[{symbols[i % len(symbols)]}] {msg}   ")
        sys.stdout.flush()
        time.sleep(0.1)
        i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 10) + "\r")
    sys.stdout.flush()

# Step 1: Create data directory
os.makedirs("/content/boltz_data", exist_ok=True)
os.chdir("/content/boltz_data/")

# Step 2: Write YAML file
yaml_content = f"""\
version: 1
sequences:
    - protein:
        id: [A]
        sequence: MVTPE
    - ligand:
        id: [B]
        ccd: SAH
"""
with open("/content/boltz_data/test.yaml", "w") as f:
    f.write(yaml_content)

# Step 3: Run boltz predict (silent)
step_msg = f"{Color.YELLOW}Downloading CCD Dataset...{Color.RESET}"
stop_event = threading.Event()
t = threading.Thread(target=loader, args=(step_msg, stop_event))
t.start()
try:
    import subprocess
    subprocess.run(
        ["boltz", "predict", "test.yaml", "--use_msa_server"],
        cwd="/content/boltz_data",
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )
    stop_event.set()
    t.join()
    print(f"[{Color.GREEN}✔{Color.RESET}] CCD Dataset Downloaded and validated.")
except Exception as e:
    stop_event.set()
    t.join()
    print(f"[{Color.RED}✘{Color.RESET}] CCD Dataset Download or validation failed: {e}")


In [ ]:
# @title Generate Parameters
%run /content/boltz_data/scripts/v1/param_gen.py

In [ ]:
# @title Run Boltz2 Engine
%run /content/boltz_data/scripts/v1/Boltz_Run.py

In [ ]:
# @title Analyse Results
%run /content/boltz_data/scripts/v1/analysis.py

In [ ]:
# @title Copy Results to Drive
import shutil, os
from google.colab import drive
from Bio.PDB import MMCIFParser, PDBIO

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BLUE = "\033[94m"
    MAGENTA = "\033[95m"
    RESET = "\033[0m"

# Mount Google Drive
drive.mount('/content/drive')

# Paths
drive_output_dir = f"/content/drive/MyDrive/Boltz2_Results/{job_name}"
local_output_path = f"/content/boltz_data/{job_name}"

# # Convert CIF to PDB
# cif_file = f"{local_output_path}/boltz_results_{job_name}/predictions/{job_name}/{job_name}_model_0.cif"
# pdb_file = f"{local_output_path}/{job_name}.pdb"

# parser = MMCIFParser(QUIET=True)
# structure = parser.get_structure("prot", cif_file)
# io = PDBIO()
# io.set_structure(structure)
# io.save(pdb_file)

# Remove old folder in Drive if exists
if os.path.exists(drive_output_dir):
    print(f"Removing existing folder {drive_output_dir}")
    shutil.rmtree(drive_output_dir)
    print("Old Drive folder removed.")

# Copy local output folder to Drive
shutil.copytree(local_output_path, drive_output_dir)
print(f"{Color.GREEN}All results copied to Google Drive: {drive_output_dir}{Color.RESET}")
# # Copy PDB file separately (optional, just in case)
# drive_pdb_file = os.path.join(drive_output_dir, os.path.basename(pdb_file))
# shutil.copy(pdb_file, drive_pdb_file)

In [ ]:
# @title Download Results (.zip)
from google.colab import files
from Bio.PDB import MMCIFParser, PDBIO
import shutil
import os

# Local output folder you want to download
local_output_path = f"/content/boltz_data/{job_name}"

# # Convert CIF to PDB
# cif_file = f"{local_output_path}/boltz_results_{job_name}/predictions/{job_name}/{job_name}_model_0.cif"
# pdb_file = f"{local_output_path}/{job_name}.pdb"

# # Parse CIF and save as PDB
# parser = MMCIFParser(QUIET=True)
# structure = parser.get_structure("prot", cif_file)
# io = PDBIO()
# io.set_structure(structure)
# io.save(pdb_file)

# Path for the zip file
zip_file = f"/content/{job_name}.zip"

# Remove previous zip if exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create zip of the entire folder
shutil.make_archive(base_name=f"/content/{job_name}", format='zip', root_dir=local_output_path)

# Download the zip file
files.download(zip_file)

# Success message
print(f"{Color.GREEN}Download successful! All results from '{job_name}' are saved in '{zip_file}'{Color.RESET}")


## Credits

**Developed by:** Atharva Tilewale & Dr. Dhaval Patel  
**Affiliation:** Gujarat Biotechnology University | Bioinformatics & Computational Biology  
**Repository:** [github.com/AtharvaTilewale/boltz2-notebook](https://github.com/AtharvaTilewale/boltz2-notebook)

---

## Citation

If you use this notebook, please cite:

**Boltz2-Notebook:**

[![Zenodo](https://img.shields.io/badge/Zenodo-Boltz2--Notebook-blue)](https://doi.org/10.5281/zenodo.21850102)

```
Tilewale, A., Patel, D. (2026). Boltz2-Notebook: A streamlined Colab-based pipeline for biomolecular structure prediction and binding affinity analysis using the Boltz2 deep learning model. Zenodo. https://doi.org/10.5281/zenodo.21850102
```

**Boltz-2:**

[![bioRxiv Boltz2](https://img.shields.io/badge/bioRxiv-Boltz2-red)](https://doi.org/10.1101/2025.06.14.659707)

```
Passaro, S., Corso, G., Wohlwend, J., et al. (2025).
Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.
bioRxiv. https://doi.org/10.1101/2025.06.14.659707
```

**Boltz-1:**

[![bioRxiv Boltz1](https://img.shields.io/badge/bioRxiv-Boltz1-orange)](https://doi.org/10.1101/2024.11.19.624167)

```
Wohlwend, J., Corso, G., Passaro, S., et al. (2024).
Boltz-1: Democratizing Biomolecular Interaction Modeling.
bioRxiv. https://doi.org/10.1101/2024.11.19.624167
```

---

## License

MIT License — See [LICENSE](https://github.com/AtharvaTilewale/boltz2-notebook/blob/main/LICENSE) for details.

---